In [6]:
#This houses the code used to create the normalized_health_dataset
import pandas as pd
from datascience import *

#Transforms strings into numeric
from sklearn.preprocessing import LabelEncoder

#Define dataset
#health_db = pd.read_csv("health_dataset.csv")
health_db = pd.read_csv("datasets/reduced_health_dataset.csv")

In [7]:
#Transform string values into numeric using label encoding
#str_columns = ["student_id", "gender", "part_time_job", "diet_quality", "parental_education_level",
#           "internet_quality", "extracurricular_participation"]

#I removed the columns I'm not using beforehand to make it easier to add more data in the future
str_columns = ["part_time_job", "diet_quality", "extracurricular_participation"]

for feature in str_columns:
    le = LabelEncoder()
    health_db[feature] = le.fit_transform(health_db[feature])

health_db.info() #Show column values
health_db.head(5) #Show dataset structure

health_db.to_csv("datasets/quant_health_dataset.csv", index = False)


<class 'pandas.DataFrame'>
RangeIndex: 1004 entries, 0 to 1003
Data columns (total 9 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   daily_study_hours              1004 non-null   float64
 1   social_media_hours             1004 non-null   float64
 2   tv_hours                       1004 non-null   float64
 3   part_time_job                  1004 non-null   int64  
 4   sleep_hours                    1004 non-null   float64
 5   diet_quality                   1004 non-null   int64  
 6   exercise_frequency_weekly      1004 non-null   float64
 7   mental_health_rating           1004 non-null   float64
 8   extracurricular_participation  1004 non-null   int64  
dtypes: float64(6), int64(3)
memory usage: 70.7 KB


In [8]:
#Fix the diet_quality scoring issue (messed up order)
#ONLY RUN WHEN NEEDED
order_fix = {
    0: 1, #Poor
    1:2, #Fair
    2:0 #Good
}

#Change the values
health_db["diet_quality"] = health_db["diet_quality"].map(order_fix)

#Save to csv (updates both quant and normalized...)
health_db.to_csv("datasets/quant_health_dataset.csv")

health_db.head()

,daily_study_hours,social_media_hours,tv_hours,part_time_job,sleep_hours,diet_quality,exercise_frequency_weekly,mental_health_rating,extracurricular_participation
0,1.4,3.1,1.3,0,8.0,0,1.0,1.0,0
1,0.0,1.2,1.1,0,8.0,1,6.0,8.0,1
2,6.9,2.8,2.3,0,4.6,2,6.0,8.0,0
3,1.0,3.9,1.0,0,9.2,0,4.0,1.0,1
4,5.0,4.4,0.5,0,4.9,1,3.0,1.0,0


In [9]:
#Dataset is not consistent enough for the model. This block will create a function
#that will treat each column as a question that impact the final rating.

#Normalization function
def value_normalize(value, min_value, max_value):
    """Function to normalize numeric values.
    Inputs:
        value: Column referneced
        min_value: Minimum value of column
        max_value: Maximum value of column
    Returns:
        normalize (decimal): Normalized value 
    """
    #Equation for normalization
    normalize = (value - min_value) / (max_value - min_value)

    #Return new value
    #Also caps the value from 0-1. No outliers
    return max(0, min (1, normalize))


#Consistant score calculator function
#More columns can be added in the future if needed
def score_calculator(row):
    """Function to calculate the mental_health_score of every row through normalization. This is to 
    replace inconsistant scoring with calculated scoring based on the same weights.
    Inputs:
        row: Each row in health_db
    Returns:
        final_score: Score based off all normalized columns that will be used
    """
    #Normalizing each column that will be used. row, min, max structure
    sleep_norm = value_normalize(row["sleep_hours"], 0, 10)
    study_norm = 1 - value_normalize(row["daily_study_hours"], 0, 8.5)
    social_media_norm = 1 - value_normalize(row["social_media_hours"], 0, 8)
    tv_norm = value_normalize(row["tv_hours"], 0, 5.4)
    job_norm = 1 - value_normalize(row["part_time_job"], 0, 1)
    diet_quality_norm = value_normalize(row["diet_quality"], 0, 2)
    exercise_norm = value_normalize(row["exercise_frequency_weekly"], 0, 6)
    extra_norm = value_normalize(row["extracurricular_participation"], 0, 1)
    
    #Weight Application to determine final score (Combined they all add to 1.0)
    #The weighted total. It creates a score for the information in each row.
    
    #My own interpretation of a healthly lifestyle and importances, weights serve as points 
    #each question is worth in a test.

    #IMPORTANT TO KNOW:
    #Above, some norms have 1- because as they increase the ratings should decrease.

    final_score = (0.15*exercise_norm + 0.25*sleep_norm + 0.15*study_norm + 
                   0.15*social_media_norm + 0.09*job_norm + 0.1*diet_quality_norm +
                   0.08*tv_norm + 0.08*extra_norm)

   
   
    #Return final result
    #On a scale of 0 - 1
    return final_score 

#Replace current ratings with new the ones calculated
health_db["mental_health_rating"] = health_db.apply(score_calculator, axis = 1)

#Save data to new csv
health_db.to_csv("datasets/normalized_health_dataset.csv", index = False)

#IMPORTANT:
#To test out the equation's performance, you need restart fastapi first
    

In [11]:
#Code to group scores in ranges for classification model
#Current 

#Read new dataset
norm_db = pd.read_csv("datasets/normalized_health_dataset.csv")

#Categroize the scores
def score_categorizer(score):
    if score < 0.5:
        return "Low"
    elif score < 0.75:
        return "Fair"
    else: 
        return "Good"

#Perform the change
norm_db["mental_health_rating"] = norm_db["mental_health_rating"].apply(score_categorizer)

#Update to new csv
norm_db.to_csv("datasets/class_normalized_health_dataset.csv", index = False)

In [10]:
#How does min max normalization work?
#It takes the lowest target value as 0, and the highest target value at 1, and then determines what the 
#others should be based on those 2 values.